<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/DecisionIntelligenceWorkshopLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Decisions with Native .NET Functions</span>
</div>

Decision Intelligence applied in this module:  
* Selecting a decision-making framework dynamically using native .NET code  
* Decision Scenario: Using a Monte Carlo Simulation to provide an estimate of decision uncertainty  
* Providing a range for Decision Uncertainty using Confidence Intervals 

Microsoft.Extensions.AI provides abstractions for intelligent AI orchestration in .NET. One useful pattern is exposing native .NET methods as `AIFunction` objects. These functions can wrap business logic, API calls, calculations, simulations, or any other capability that is better handled by code than by prompting alone.

What makes native .NET functions useful? GenAI models can reason through instructions, but real decision workflows often need trusted deterministic logic. A model can discuss confidence intervals, but a C# Monte Carlo simulation can actually calculate one. A model can recommend a framework, but a native function can enforce a business rule that selects the approved framework for a given decision type.

For example, imagine you want to prepare a great Thanksgiving dinner and you ask a GenAI cooking application to create a recipe. It may produce a delicious plan, but it may also use ingredients or cooking appliances you do not own. It is better when the AI workflow can call deterministic code that checks available ingredients, time availability, kitchen appliances, and allergy preferences. The same pattern applies to Decision Intelligence: native .NET functions let the AI workflow use trusted logic and data.

In Microsoft.Extensions.AI, `AIFunctionFactory.Create(...)` can wrap .NET methods as first-class AI functions. These functions can be invoked directly with `AIFunctionArguments`, or they can be registered as tools in `ChatOptions.Tools` for model-driven function calling.

### Step 1 - Initialize Configuration Builder & Build the AI Extensions Orchestration 

Execute the next two cells to:
* Use the Configuration Builder to load the API secrets.  
* Use the API configuration to build a Microsoft.Extensions.AI `IChatClient` orchestration.

In [ ]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.8"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.8"
#r "nuget: System.Text.Json, 10.0.8"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

In [ ]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.6.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

#pragma warning disable OPENAI001

if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);
    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(openAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);
    chatClient = openAIClient.GetChatClient(openAIModelId).AsIChatClient();
}

#pragma warning restore OPENAI001

static string GetFunctionResultText(object? result)
{
    if (result is null)
    {
        return string.Empty;
    }

    if (result is JsonElement jsonElement)
    {
        return jsonElement.ValueKind == JsonValueKind.String
            ? jsonElement.GetString() ?? string.Empty
            : jsonElement.ToString();
    }

    return result.ToString() ?? string.Empty;
}

### Step 2 - Create a Simple Native .NET Function with AI Extensions

Execute the cell below to create a very simple native .NET function using a C# inline method. Notice the function takes no parameters. It retrieves the name of a productivity decision framework to use. In this case, that return name is hard-coded to "Price's Law".

The function returns instantly because it is not calling a GenAI service. The point is to show how Microsoft.Extensions.AI can wrap deterministic .NET logic as a first-class `AIFunction`.

In [ ]:
// Create an AIFunction from an inline C# method.
var nameOfProductivityFramework = AIFunctionFactory.Create(
    (Func<string>)(() => "Price's Law"),
    new AIFunctionFactoryOptions
    {
        Name = "GetNameOfProductivityFramework",
        Description = "Retrieves the name of the Productivity Framework to use."
    });

// Invoke the function directly with AI Extensions.
var response = await nameOfProductivityFramework.InvokeAsync(new AIFunctionArguments());
Console.WriteLine(GetFunctionResultText(response));

### Step 3 - Create a Native .NET Function with Dynamic Parameters

Execute the cell below to create a native .NET function that takes a parameter as input. This shows that C# native functions can have different execution paths. The execution paths can obviously be quite complex. Basically, any C# logic flow will work.

In [ ]:
// Create an AIFunction from an inline C# method with parameters.
var nameOfProductivityFrameworkByType = AIFunctionFactory.Create(
    (Func<string, string>)(typeOfProductivity => typeOfProductivity == "Sales" ? "Price's Law" : "Pareto Principle"),
    new AIFunctionFactoryOptions
    {
        Name = "GetNameOfProductivityFramework",
        Description = "Retrieves the name of the Productivity Framework to use."
    });

// Pass the "Sales" parameter to the function.
var salesResponse = await nameOfProductivityFrameworkByType.InvokeAsync(new AIFunctionArguments
{
    ["typeOfProductivity"] = "Sales"
});
Console.WriteLine(GetFunctionResultText(salesResponse));

// Pass the "Other" parameter to the function.
var otherResponse = await nameOfProductivityFrameworkByType.InvokeAsync(new AIFunctionArguments
{
    ["typeOfProductivity"] = "Other"
});
Console.WriteLine(GetFunctionResultText(otherResponse));

### Step 4 - Use Native .NET Functions To Simulate the Uncertainty of a Decision  

> 📜 **_"The best way to predict the future is to simulate it. And the best way to simulate it is with Monte Carlo."_**
>
> -- <cite>Nassim Nicholas Taleb (Lebanese-American essayist, scholar, best known for his work on probability)</cite>

For more complex decisions, native .NET functions can use statistics, advanced probabilistic algorithms, analytics, machine learning, AI, and other approaches that have been relied on in software for decades. One such method is Monte Carlo Simulation. These powerful Monte Carlo simulation techniques are used everywhere: risk management, sports gambling, medical decision-making, game theory, energy market forecasting, and more. In simple terms, a Monte Carlo simulation is a series of many runs testing different plausible parameters. Running a Monte Carlo simulation many times results in an output of a plausible range.  

A simple use case for a Monte Carlo simulation is to provide a realistic range for an average probability. Imagine you want to illustrate the uncertainty of a decision that you have calculated to be 70% successful. On "average" the probability of success can be interpreted as 70%. However, what is the range of possible successes if that 70% decision model is run 100x? A Monte Carlo simulation can help solve that answer.

Run the cell below to create a new native .NET function that will take in the confidence parameter and output a simple string with the lower and upper bounds of a 95% confidence interval. A 95% Confidence Interval output will tell us if we execute this calculated decision 100x that has a 70% success probability, what is the realistic range of success in those 100 decisions.

In [ ]:
string RetrieveConfidenceIntervalMonteCarlo(
    [Description("Claimed Probability Percentage")] int probability)
{
    const int NUMBEROFSIMULATIONS = 100000; // 100,000 simulations, make this smaller for faster results
    Console.WriteLine($"Simulating {NUMBEROFSIMULATIONS:n0} iterations with a claimed decision confidence of {probability}%...");

    var random = new Random(); // Add seed for reproducibility
    var bootstrapConfidenceScores = new List<double>();
    for (int i = 0; i != NUMBEROFSIMULATIONS; i++) // Bootstrap Simulations (bootstrap estimates)
    {
        var bootstrapSample = new List<double>();
        for (int j = 0; j != 100; j++)
        {
            var randomIndex = random.Next(0, 100);

            if (randomIndex < probability)
            {
                bootstrapSample.Add(1);
            }
        }

        bootstrapConfidenceScores.Add(bootstrapSample.Count);
    }

    // Sort the confidence scores to calculate the percentiles
    var bootstrapConfidenceScoresSorted = bootstrapConfidenceScores.OrderBy(a => a).ToList();
    // Calculate the 2.5% and 97.5% percentiles
    var lowerPercentileIndex = Convert.ToInt32(0.025 * NUMBEROFSIMULATIONS);
    var topPercentileIndex = Convert.ToInt32(0.975 * NUMBEROFSIMULATIONS);

    var lowerPercentile = Math.Round(bootstrapConfidenceScoresSorted[lowerPercentileIndex], 3);
    var upperPercentile = Math.Round(bootstrapConfidenceScoresSorted[topPercentileIndex], 3);

    var confidenceUncertaintyRange = $"95% Confidence Interval of a {probability}% success model: {lowerPercentile} to {upperPercentile} successful decision outcomes of 100 decisions made.";
    return confidenceUncertaintyRange;
}

Run the cell below to wrap the native .NET method as an `AIFunction` and invoke it directly. This will run a Monte Carlo Simulation with 100,000 simulations of a decision with a confidence (probability) of 70% being run 100x. Basically, this will provide the uncertainty range if you had made 100 decisions of the same 70% success decision.

In [ ]:
var retrieveConfidenceInterval = AIFunctionFactory.Create(
    (Func<int, string>)RetrieveConfidenceIntervalMonteCarlo,
    new AIFunctionFactoryOptions
    {
        Name = "RetrieveConfidenceIntervalMonteCarlo",
        Description = "Generates Confidence Interval from provided Confidence Percentage"
    });

// Change the probability to another integer and invoke the function, to see other Confidence Intervals.
var confidenceIntervalRange70 = await retrieveConfidenceInterval.InvokeAsync(new AIFunctionArguments
{
    ["probability"] = 70
});
Console.WriteLine(GetFunctionResultText(confidenceIntervalRange70));